In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['figure.figsize'] = 12,6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

# 데이터 전처리 관련 ####################################################
# 결측치 처리
from sklearn.impute import SimpleImputer
# 표준화
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
# 인코더
from sklearn.preprocessing import LabelEncoder
# 원핫 인코더
from sklearn.preprocessing import OneHotEncoder

# 학습 모델 성능 관련 ####################################################
# 원하는 비율로 데이터를 나누기 위해
from sklearn.model_selection import train_test_split
# K-Fold 교차 검증
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
# 학습곡선
from sklearn.model_selection import learning_curve
# 하이퍼 파라미터 튜닝
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint
import optuna
from optuna.samplers import TPESampler # 데이터를 랜덤샘플링하기 위함
from optuna.pruners import MedianPruner
optuna.logging.set_verbosity(optuna.logging.WARNING) # 수치가 떨어지면 경고로그가 뜨는데 그거를 막아줌

# 모델 성능평가 #############################################
# 회귀용
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import r2_score
# 분류용
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

from sklearn.metrics import roc_curve
from sklearn.metrics import auc
from sklearn.metrics import roc_auc_score
from sklearn.metrics import ConfusionMatrixDisplay

# 피처 선택 ################################################
from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import RFE
from sklearn.inspection import permutation_importance

# 학습모델 ##################################################
#분류
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.naive_bayes import GaussianNB
from catboost import CatBoostClassifier

#회귀
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import BayesianRidge
from catboost import CatBoostRegressor

# 결정트리를 시각화할 수 있는 라이브러리
from sklearn.tree import plot_tree

# 차원축소
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.manifold import TSNE

# 연관규칙 학습
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import fpgrowth
from mlxtend.frequent_patterns import association_rules

# 군집
from sklearn.cluster import KMeans
from sklearn.cluster import DBSCAN
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.mixture import GaussianMixture
from sklearn.cluster import MeanShift, estimate_bandwidth

# 파이프라인
from sklearn.pipeline import Pipeline

# KDE를 그리기 위한 통계값을 구할 수 있는 함수
from scipy.stats import gaussian_kde

# 피어슨 상관 계수 (연속형 수치형 데이터 vs 연속형 수치형 데이터)
from scipy.stats import pearsonr
# 카이제곱 검증 (범주형 데이터 vs 범주현 데이터, 순위 x)
from scipy.stats import chi2_contingency
# 스피어만 상관계수 (범주형 데이터 vs 범주형 데이터, 순위 O)
from scipy.stats import spearmanr
# 포인트 이분 상관계수 (범주형 데이터 vs 연속형 수치형 데이터)
from scipy.stats import pointbiserialr

# 객체를 파일에 저장
import pickle

# 불필요한 경고 뜨지 않게
import warnings
warnings.filterwarnings('ignore')

### 데이터 불러오기

In [2]:
df = pd.read_csv('data/ThoraricSurgery2.csv')
df

,PRE4,PRE5,PRE7,PRE8,PRE9,PRE10,PRE11,PRE17,PRE19,PRE25,...,DGN_DGN5,DGN_DGN6,DGN_DGN8,PRE6_PRZ0,PRE6_PRZ1,PRE6_PRZ2,PRE14_OC11,PRE14_OC12,PRE14_OC13,PRE14_OC14
0,2.88,2.16,0,0,0,1,1,0,0,0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
1,3.40,1.88,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
2,2.76,2.08,0,0,0,1,0,0,0,0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
3,3.68,3.04,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
4,2.44,0.96,0,1,0,1,1,0,0,0,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
465,3.88,2.12,0,0,0,1,0,0,0,0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
466,3.76,3.12,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
467,3.04,2.08,0,0,0,1,0,0,0,0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
468,1.96,1.68,0,0,0,1,1,0,0,0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0


In [3]:
# 결과데이터의 도수 분포를 확인해본다.
df['Risk1Yr'].value_counts()

Risk1Yr
0    400
1     70
Name: count, dtype: int64

In [4]:
# 연속형 수치형 데이터 컬럼 목록
a1 = ['PRE4','PRE5','AGE']
# 컬럼 이름들을 가져온다
column_name_list = df.columns.to_list()
# 결과 데이터는 제외한다.
column_name_list.remove('Risk1Yr')
# p value를 담을 리스트
p_value_list = []
# 컬럼의 목록을 통해 반복 수행
for c1 in column_name_list :
    # 만약 연속형 수치형 데이터 라면~
    if c1 in a1 :    
        _, p_value = pointbiserialr(df[c1], df['Risk1Yr'])
    # 만약 번주형 데이터 라면~
    else :
        crosstab = pd.crosstab(df[c1], df['Risk1Yr'])
        _, p_value, _, _ = chi2_contingency(crosstab)

    # 리스트에 담아준다.
    p_value_list.append(p_value)

p_value_df = pd.DataFrame(p_value_list, index=column_name_list, columns=['P_value'])
p_value_df.sort_values('P_value', ascending=False, inplace=True)
p_value_df

,P_value
PRE19,1.000000
DGN_DGN1,1.000000
PRE32,1.000000
DGN_DGN4,1.000000
PRE14_OC12,0.953634
DGN_DGN6,0.892581
PRE25,0.757314
DGN_DGN8,0.687459
PRE6_PRZ1,0.604989
AGE,0.400102


In [5]:
drop_index = p_value_df.query('P_value >= 0.05').index
drop_index

Index(['PRE19', 'DGN_DGN1', 'PRE32', 'DGN_DGN4', 'PRE14_OC12', 'DGN_DGN6',
       'PRE25', 'DGN_DGN8', 'PRE6_PRZ1', 'AGE', 'PRE5', 'PRE7', 'PRE4', 'PRE8',
       'PRE6_PRZ2', 'PRE6_PRZ0', 'DGN_DGN2', 'PRE30', 'PRE11', 'PRE14_OC13',
       'PRE10'],
      dtype='str')

In [6]:
df.drop(drop_index, axis=1, inplace=True)
df

,PRE9,PRE17,Risk1Yr,DGN_DGN3,DGN_DGN5,PRE14_OC11,PRE14_OC14
0,0,0,0,0.0,0.0,0.0,1.0
1,0,0,0,1.0,0.0,0.0,0.0
2,0,0,0,1.0,0.0,1.0,0.0
3,0,0,0,1.0,0.0,1.0,0.0
4,0,0,1,1.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...
465,0,0,0,0.0,0.0,0.0,0.0
466,0,0,0,1.0,0.0,1.0,0.0
467,0,0,0,1.0,0.0,0.0,0.0
468,0,0,0,1.0,0.0,0.0,0.0


In [7]:
df.to_csv('data/ThoraricSurgery3.csv', index=False)